In [1]:
import joblib
import pandas as pd 
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [2]:
X_train = joblib.load("../models/data/X_train.pkl")
y_train = joblib.load("../models/data/y_train.pkl")
print("Training Data loaded")
print("X_train shape: ", X_train.shape)

Training Data loaded
X_train shape:  (960, 21)


In [3]:
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state= 42
)

In [4]:
smote = SMOTE(random_state=42)


In [5]:
models = {
    "LogisticRegression" : {
        "model" : Pipeline([("smote", smote), ("classifier", LogisticRegression())]),
        "params" : {
            "classifier__C" : [0.01, 0.1, 1, 10],
            "classifier__solver" : ["liblinear", "lbfgs"],
            "classifier__max_iter" : [1000]
        }
    },
    "RandomForest": {
        "model": Pipeline([("smote", smote), ("classifier", RandomForestClassifier())]),
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__min_samples_leaf": [1, 2]
        }
    },
    "XGBoost": {
        "model": Pipeline([("smote", smote), ("classifier", XGBClassifier(eval_metric="mlogloss"))]),
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [3, 5, 7],
            "classifier__learning_rate": [0.01, 0.1],
            "classifier__subsample": [0.8, 1.0]
        }
    },
    "SVM": {
        "model": Pipeline([("smote", smote), ("classifier", SVC())]),
        "params": {
            "classifier__C": [0.1, 1, 10],
            "classifier__kernel": ["linear", "rbf"],
            "classifier__gamma": ["scale", "auto"]
        }
    },
    "KNN": {
        "model": Pipeline([("smote", smote), ("classifier", KNeighborsClassifier())]),
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"]
        }
    }
}

In [6]:
best_models = {}
results = []

In [7]:
for model_name , mp in models.items():
    print('='*45)
    print(f"Training {model_name}")
    print('='*45)

    grid_search = GridSearchCV(
        estimator=mp["model"],
        param_grid=mp["params"],
        cv=kfold,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    best_score = grid_search.best_score_
    best_params = grid_search.best_params_
    best_models[model_name] = best_model

    results.append({
        "Model": model_name,
        "Best Score": best_score,
        "Best Params": best_params
    })

    print(f"\nBest Accuracy: {best_score:.4f}")
    print("\nBest Parameters:")
    print(best_params)
    print("\nModel Training Completed")

Training LogisticRegression
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Accuracy: 0.8990

Best Parameters:
{'classifier__C': 10, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs'}

Model Training Completed
Training RandomForest
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Accuracy: 0.9698

Best Parameters:
{'classifier__max_depth': 20, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}

Model Training Completed
Training XGBoost
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Accuracy: 0.9656

Best Parameters:
{'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 200, 'classifier__subsample': 0.8}

Model Training Completed
Training SVM
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best Accuracy: 0.9583

Best Parameters:
{'classifier__C': 10, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}

Model Tr

In [8]:
results_df = pd.DataFrame(results)

In [9]:
print("Final Result")
results_df

Final Result


,Model,Best Score,Best Params
0,LogisticRegression,0.898958,"{'classifier__C': 10, 'classifier__max_iter': ..."
1,RandomForest,0.969792,"{'classifier__max_depth': 20, 'classifier__min..."
2,XGBoost,0.965625,"{'classifier__learning_rate': 0.1, 'classifier..."
3,SVM,0.958333,"{'classifier__C': 10, 'classifier__gamma': 'sc..."
4,KNN,0.939583,"{'classifier__metric': 'manhattan', 'classifie..."


In [10]:
best_model_name = results_df.sort_values(
    by="Best Score",
    ascending=False
).iloc[0]["Model"]

best_model = best_models[best_model_name]

print("\n=================================================")
print(f"BEST MODEL: {best_model_name}")
print("=================================================")


BEST MODEL: RandomForest


In [11]:
for model_name, model in best_models.items():

    filename = f"../models/{model_name}.pkl"

    joblib.dump(model, filename)

print("\nAll models saved successfully")


All models saved successfully


In [12]:
results_df.to_csv(
    "../models/model_comparison_results.csv",
    index=False
)

print("\nResults CSV saved successfully")


Results CSV saved successfully


In [13]:
print("""

Generated Files:
----------------
1. best_model.pkl
2. LogisticRegression.pkl
3. RandomForest.pkl
4. XGBoost.pkl
5. SVM.pkl
6. KNN.pkl
7. model_comparison_results.csv
At ../models/
""")



Generated Files:
----------------
1. best_model.pkl
2. LogisticRegression.pkl
3. RandomForest.pkl
4. XGBoost.pkl
5. SVM.pkl
6. KNN.pkl
7. model_comparison_results.csv
At ../models/

